# Практика: сравнение кривых эффекта признака

Модуль прошел семейство методов «эффекта признака» от одного объекта к модели целиком: CPO → ICE → PDP → ALE. В этой тетради они ставятся рядом на одной модели и одних данных: с чего модуль начался (две разные важности признаков одного случайного леса), как выглядят все кривые одного признака на одном графике, совпадают ли ранжирования признаков по разным методам и сколько каждый метод стоит в прогнозах модели.

Все методы считаются средствами `scikit-learn` и руками на `numpy`/`pandas`, без дополнительных пакетов: PDP, ICE и c-ICE есть в `sklearn.inspection`, а CPO-оценка, d-ICE, M-plot и ALE пишутся в несколько строк. Пакеты, где эти методы есть готовыми, перечислены в шаге модуля «Чем это считать: библиотеки».

В каждом задании есть ячейка с `...` — ее нужно дописать. Заданий в тренажере по этой тетради нет.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## Данные и модель

Набор California Housing встроен в `sklearn`: 20 640 районов Калифорнии из переписи 1990 года, восемь признаков района, целевая переменная — медианная стоимость дома в районе в сотнях тысяч долларов. Признаки: `MedInc` — медианный доход (десятки тысяч долларов), `HouseAge` — медианный возраст дома, `AveRooms` и `AveBedrms` — среднее число комнат и спален на дом, `Population` — население района, `AveOccup` — среднее число жильцов на дом, `Latitude` и `Longitude` — широта и долгота.

Делим данные на обучающую и тестовую части (80% и 20%, `random_state=42`) и обучаем градиентный бустинг `HistGradientBoostingRegressor` с параметрами по умолчанию. Для методов этой тетради модель — черный ящик: от нее нужен только прогноз `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"R^2 модели на тестовых данных: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

## 1. Две важности одного случайного леса

Введение модуля начиналось с того, что важность признаков одного и того же случайного леса, посчитанная двумя способами, дает разные картины. Воспроизведем это. К признакам добавим два заведомо бесполезных: `rnd_bin` — случайный бинарный признак, `rnd_num` — гауссов шум.

**Задание 1.** Обучите `RandomForestRegressor(n_estimators=100, random_state=42)` и посчитайте две важности: встроенную, по уменьшению неоднородности в узлах деревьев (MDI, `feature_importances_`), и перестановочную на тестовых данных (`permutation_importance`, `n_repeats=5`, `random_state=42`). Постройте их рядом.

In [ ]:
rng = np.random.RandomState(42)
X_rnd = X.assign(rnd_bin=rng.randint(0, 2, len(X)), rnd_num=rng.randn(len(X)))
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_rnd, y, test_size=0.2, random_state=42)
forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(Xr_train, yr_train)
print(f"R^2 леса на тестовых данных: {forest.score(Xr_test, yr_test):.3f}")

mdi = ...        # ваш код: встроенная важность леса
perm = ...       # ваш код: permutation_importance на тестовых данных
perm_mean = ...  # средняя перестановочная важность
perm_std = ...   # ее стандартное отклонение по 5 перестановкам

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
order = mdi.sort_values().index
axes[0].barh(order, mdi[order], color=BLUE)
axes[0].set_title("MDI: уменьшение неоднородности на обучающих данных", fontsize=10)
order = perm_mean.sort_values().index
axes[1].barh(order, perm_mean[order], xerr=perm_std[order], color=BLUE, ecolor=INK)
axes[1].set_title("перестановочная: падение R^2 на тестовых данных", fontsize=10)
fig.suptitle("Две важности признаков одного случайного леса, California Housing с двумя случайными признаками")
fig.tight_layout()
plt.show()
pd.DataFrame({"MDI": mdi, "перестановочная": perm_mean}).sort_values("MDI", ascending=False).round(3)

**Место для ваших выводов:** совпадают ли первые четыре признака в двух рейтингах? Какую долю суммы важностей получает `MedInc` в каждом из них? Какую важность получил шумовой признак `rnd_num`?

## 2. Все кривые одного признака

Дальше — градиентный бустинг `model` и признак `Latitude`. Широта сильно связана с долготой, и на ней видно, чем методы расходятся. Ниже готовы функции из предыдущих практик модуля: ICE, PDP и ALE.

**Задание 2.** Допишите M-plot — средний прогноз модели по объектам, у которых широта лежит в окне $\pm 0.25$ градуса вокруг точки сетки. Затем постройте на одном графике центрированные кривые PDP, M-plot и ALE, а под ними — 50 кривых c-ICE.

In [ ]:
def ice_curves(model, X, feature, grid):
    X_rep = pd.concat([X] * len(grid), ignore_index=True)
    X_rep[feature] = np.repeat(grid, len(X))
    return model.predict(X_rep).reshape(len(grid), len(X)).T


def pdp(model, X, feature, grid):
    return ice_curves(model, X, feature, grid).mean(axis=0)


def ale(model, X, feature, K=20):
    x = X[feature].values
    z = np.unique(np.quantile(x, np.linspace(0, 1, K + 1)))
    k = np.clip(np.searchsorted(z, x, side="left") - 1, 0, len(z) - 2)
    lo, hi = X.copy(), X.copy()
    lo[feature], hi[feature] = z[k], z[k + 1]
    n_k = np.bincount(k, minlength=len(z) - 1)
    delta = np.bincount(k, weights=model.predict(hi) - model.predict(lo), minlength=len(z) - 1) / n_k
    A = np.concatenate([[0.0], np.cumsum(delta)])
    mid = (A[:-1] + A[1:]) / 2
    return z, A - (mid * n_k).sum() / n_k.sum()


def m_plot(model, X, feature, grid, width):
    return ...  # ваш код: для каждой точки g — средний прогноз объектов с |X[feature] - g| < width


print("Корреляция широты и долготы:", X["Latitude"].corr(X["Longitude"]).round(3))

In [ ]:
feature = "Latitude"
X_cmp = X_test.sample(1000, random_state=0)
grid = np.linspace(X_cmp[feature].quantile(0.05), X_cmp[feature].quantile(0.95), 20)
curve_pdp = pdp(model, X_cmp, feature, grid)
curve_m = m_plot(model, X_test, feature, grid, width=0.25)
z, curve_ale = ale(model, X_cmp, feature)
ice_c = ice_curves(model, X_cmp.iloc[:50], feature, grid)
ice_c = ice_c - ice_c[:, [0]]

fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
axes[0].plot(grid, curve_pdp - curve_pdp.mean(), color=ORANGE, lw=2, label="PDP")
axes[0].plot(grid, curve_m - curve_m.mean(), color=AQUA, lw=2, label="M-plot")
axes[0].plot(z, curve_ale, color=BLUE, lw=2, label="ALE")
axes[0].set_ylabel("эффект, центрированный")
axes[0].legend()
axes[1].plot(grid, ice_c.T, color=BLUE, alpha=0.35, lw=1)
axes[1].set_ylabel("c-ICE")
axes[1].set_xlabel("Latitude")
axes[1].set_xlim(grid[0], grid[-1])
fig.suptitle("Кривые эффекта широты: California Housing, градиентный бустинг")
fig.tight_layout()
plt.show()
in_grid = (z >= grid[0]) & (z <= grid[-1])
print("Размах на сетке: PDP", np.ptp(curve_pdp).round(3), " M-plot", np.ptp(curve_m).round(3), " ALE", np.ptp(curve_ale[in_grid]).round(3))

**Место для ваших выводов:** у какой кривой размах по широте самый большой и почему? Что добавляют к трем средним кривым c-ICE?

## 3. Совпадают ли ранжирования

**Задание 3.** Для градиентного бустинга упорядочьте восемь признаков тремя способами: по PDP-based Feature Importance (стандартное отклонение PDP на сетке из 20 точек между 5-м и 95-м процентилями), по размаху ALE и по перестановочной важности на тестовых данных. Посчитайте ранговую корреляцию Спирмена между каждой парой.

In [ ]:
X_rank = X_test.sample(1000, random_state=0)
scores = {}
for f in X.columns:
    g = np.linspace(X_rank[f].quantile(0.05), X_rank[f].quantile(0.95), 20)
    scores[f] = ...  # ваш код: словарь с PDP-FI и размахом ALE
scores = pd.DataFrame(scores).T
scores["перестановочная"] = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42).importances_mean
print(scores.round(3).sort_values("PDP-FI", ascending=False))
print()
print(scores.corr(method="spearman").round(3))

**Место для ваших выводов:** насколько согласны три ранжирования? Какой признак переставлен сильнее всего и чем это объяснить?

## 4. Стоимость в прогнозах модели

**Задание 4.** Посчитайте число прогнозов модели для PDP и для ALE на всех тестовых строках при $K = 10$, $50$ и $100$ точках сетки или интервалах и сравните отношение с формулой урока $K/2$.

In [ ]:
class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def predict(self, X):
        self.rows += len(X)
        return self.model.predict(X)


rows = {}
for K in [10, 50, 100]:
    c_pdp, c_ale = CountingModel(model), CountingModel(model)
    ...  # ваш код: PDP по сетке из K точек и ALE с K интервалами через счетчики
    rows[K] = {"PDP": c_pdp.rows, "ALE": c_ale.rows, "PDP / ALE": c_pdp.rows / c_ale.rows, "K / 2": K / 2}
pd.DataFrame(rows).T

## Итог: какой метод для какого вопроса

| вопрос | метод | что помнить |
|---|---|---|
| как модель ведет себя на одном объекте при изменении признака | CPO и CPO-оценка | вывод локальный; равномерная сетка по диапазону с длинным хвостом спрашивает модель там, где данных нет |
| одинаков ли эффект признака у разных объектов | ICE, c-ICE | при тысячах объектов пучок нечитаем; c-ICE сравнивает формы эффекта |
| где по оси признака возникает взаимодействие | d-ICE | у деревьев только конечные разности; готовой реализации в Python нет |
| средний эффект признака, признаки слабо связаны | PDP, PDP-based FI | смотреть на диапазон оси; плоский PDP не означает отсутствия эффекта |
| средний эффект признака, признаки связаны | ALE | зависит от числа интервалов $K$; одномерный ALE не видит взаимодействий |